In [1]:
import os
import time
from matplotlib import pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from scipy.io import arff
from ucimlrepo import fetch_ucirepo
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from scripts.train_model import circuit_training, train_five_times
from scr.qsvdd_core.data_loader import QuantumDataLoader
from scripts.test_model import test, mean_auc, best_batch, save_test_results

In [2]:
np.random.seed(42)

In [3]:
n_train = 0 ; latent_dim = 3
# num_params_conv = 375
cost_func = 'svdd'

# Breast Cancer anomaly Detection

In [4]:
print("="*60)
print("Loading and processing the Breast Cancer dataset...")
print("="*60)

# 1. Fetch the dataset
breast_cancer = fetch_ucirepo(id=17)
X_bc_raw = breast_cancer.data.features
y_bc_raw = breast_cancer.data.targets

Loading and processing the Breast Cancer dataset...


In [36]:
print(breast_cancer.data.features.head())

   radius1  texture1  perimeter1   area1  smoothness1  compactness1  \
0    17.99     10.38      122.80  1001.0      0.11840       0.27760   
1    20.57     17.77      132.90  1326.0      0.08474       0.07864   
2    19.69     21.25      130.00  1203.0      0.10960       0.15990   
3    11.42     20.38       77.58   386.1      0.14250       0.28390   
4    20.29     14.34      135.10  1297.0      0.10030       0.13280   

   concavity1  concave_points1  symmetry1  fractal_dimension1  ...  radius3  \
0      0.3001          0.14710     0.2419             0.07871  ...    25.38   
1      0.0869          0.07017     0.1812             0.05667  ...    24.99   
2      0.1974          0.12790     0.2069             0.05999  ...    23.57   
3      0.2414          0.10520     0.2597             0.09744  ...    14.91   
4      0.1980          0.10430     0.1809             0.05883  ...    22.54   

   texture3  perimeter3   area3  smoothness3  compactness3  concavity3  \
0     17.33      184.60 

In [6]:
X_bc_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 30 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   radius1             569 non-null    float64
 1   texture1            569 non-null    float64
 2   perimeter1          569 non-null    float64
 3   area1               569 non-null    float64
 4   smoothness1         569 non-null    float64
 5   compactness1        569 non-null    float64
 6   concavity1          569 non-null    float64
 7   concave_points1     569 non-null    float64
 8   symmetry1           569 non-null    float64
 9   fractal_dimension1  569 non-null    float64
 10  radius2             569 non-null    float64
 11  texture2            569 non-null    float64
 12  perimeter2          569 non-null    float64
 13  area2               569 non-null    float64
 14  smoothness2         569 non-null    float64
 15  compactness2        569 non-null    float64
 16  concavity2         

In [7]:
y_bc_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 1 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   Diagnosis  569 non-null    str  
dtypes: str(1)
memory usage: 4.6 KB


In [5]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import Normalizer

def prepare_pca_state(X, n_components=16):
        """
        Para State Preparation (Amplitude Encoding).
        Reduz para n_components (ex: 2^n_qubits) e aplica Norma L2.
        """
        std_scaler = StandardScaler()
        normalizer = Normalizer()

        X_std = std_scaler.fit_transform(X)
        pca = PCA(n_components=n_components)
        X_pca = pca.fit_transform(X_std)

        # Normalização para que a soma dos quadrados seja 1 (vetor de estado válido)
        X_state = normalizer.fit_transform(X_pca)

        print(
            f"State Prep: {X.shape[1]} -> {n_components} features. Var: {pca.explained_variance_ratio_.sum():.2%}"
        )
        return X_state

In [6]:
y_bc = y_bc_raw.iloc[:, 0].apply(lambda x: 1 if x == 'M' else 0).values

loader = QuantumDataLoader(n_qubits=4)
scaler = StandardScaler()

import pandas as pd
df_bc = pd.DataFrame(X_bc_raw)
df_bc['Class'] = y_bc

X_data = scaler.fit_transform(X_bc_raw)
X_data = prepare_pca_state(X_data, n_components=16)
y = y_bc

# 4. Identify indices for each class
normal_indices = np.where(y == 0)[0]
abnormal_indices = np.where(y == 1)[0]
np.random.seed(42)

# 5. Training Set (One-Class: 200 normal samples)
X_train_normal_indices = np.random.choice(normal_indices, 250, replace=False)
X_train = X_data[X_train_normal_indices]
Y_train = y[X_train_normal_indices]

# 6. Test Set (Balanced: 50 normal + 50 anomalies)
remaining_normal_indices = list(set(normal_indices) - set(X_train_normal_indices))

X_test_normal_indices = np.random.choice(remaining_normal_indices, 50, replace=False)
X_test_abnormal_indices = np.random.choice(abnormal_indices, 50, replace=False)

X_test_normal = X_data[X_test_normal_indices]
y_test_normal = y[X_test_normal_indices]

X_test_abnormal = X_data[X_test_abnormal_indices]
y_test_abnormal = y[X_test_abnormal_indices]

# Final test set (Mix: 100 samples)
X_test = np.concatenate((X_test_normal, X_test_abnormal), axis=0)
Y_test = np.concatenate((y_test_normal, y_test_abnormal), axis=0)

print(f"X_train shape (Normal): {X_train.shape}")
print(f"Y_train shape: {Y_train.shape}")
print(f"X_test shape (Mix): {X_test.shape}")
print(f"Y_test shape: {Y_test.shape}")

State Prep: 30 -> 16 features. Var: 98.92%
X_train shape (Normal): (250, 16)
Y_train shape: (250,)
X_test shape (Mix): (100, 16)
Y_test shape: (100,)


In [7]:
loader = QuantumDataLoader(n_qubits=4)

X_quantum = X_data

print(f"Features for the circuit: {X_quantum.shape}")
print(f"Labels: {y.shape}")

Features for the circuit: (569, 16)
Labels: (569,)


# ALOI Dataset

In [14]:
data, meta = arff.loadarff('../data/ALOI_withoutdupl_norm.arff')
df = pd.DataFrame(data)

# Pegamos da primeira coluna até a 'att27' (índice 0 até 26)
# O fatiamento :27 pega os índices de 0 a 26.
X = df.iloc[:, :27].values

# 3. Tratar o Label (y)
# A coluna de label chama-se 'outlier'
y_raw = df['outlier']

# Converter bytes para string e depois para binário (yes=1, no=0)
y = y_raw.apply(lambda x: x.decode('utf-8').lower() if isinstance(x, bytes) else str(x).lower())
y = np.where(y == 'yes', 1, 0)

# 4. Resultados Finais
print(f"{'='*30}")
print(f"DATASET ALOI CARREGADO")
print(f"{'='*30}")
print(f"Instâncias: {X.shape[0]}")
print(f"Atributos (Features): {X.shape[1]} (att1 até att27)")
print(f"Anomalias (Outliers): {np.sum(y)}")
print(f"Proporção de Outliers: {np.mean(y)*100:.2f}%")
print(f"{'='*30}")

DATASET ALOI CARREGADO
Instâncias: 49534
Atributos (Features): 27 (att1 até att27)
Anomalias (Outliers): 1508
Proporção de Outliers: 3.04%


In [17]:
def prepare_aloi_classic(df):
    # No ALOI, as colunas são 'att1'...'att27', o label é 'outlier' e tem o 'id'
    # Selecionamos as 27 colunas de atributos
    X = df.iloc[:, :27].values

    # Tratamos o label 'outlier' para 0 e 1
    y_raw = df['outlier'].apply(lambda x: x.decode('utf-8').lower() if isinstance(x, bytes) else str(x).lower())
    y = np.where(y_raw == 'yes', 1, 0)

    # Aplicar o Scaler (opcional para ALOI, mas mantém consistência com seu projeto)
    scaler = StandardScaler()
    X_classic = scaler.fit_transform(X)

    X_padded = np.pad(X_classic, ((0, 0), (0, 5)), mode='constant', constant_values=0)
    print(f"Shape para o QSVDD: {X_padded.shape}") # (49534, 32)
    # troca x_classic por x_padded no return se for quantum_algorithm
    return X_padded, y


# Executando a preparação
X_quantum, y = prepare_aloi_classic(df)

print(f"Features para o modelo: {X_data.shape}")
print(f"Labels: {y.shape}")
print(f"Total de Anomalias: {np.sum(y)}")

Shape para o QSVDD: (49534, 32)
Features para o modelo: (49534, 32)
Labels: (49534,)
Total de Anomalias: 1508


# Credit Card

In [39]:
file_path = os.path.join("..", "data", "creditcard.csv")
df = pd.read_csv(file_path)

print("Dataset loaded successfully!\n")
print("First 5 records:\n", df.head())

Dataset loaded successfully!

First 5 records:
    Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.14126

In [40]:
tmp = df[['Amount','Class']].copy()
class_0 = tmp.loc[tmp['Class'] == 0]['Amount']
class_1 = tmp.loc[tmp['Class'] == 1]['Amount']

In [41]:
loader = QuantumDataLoader()

X_quantum, y = loader.prepare_fraud_data(df)

print(f"Features for the circuit: {X_quantum.shape}")
print(f"Labels: {y.shape}")

Features for the circuit: (284807, 32)
Labels: (284807,)


# Preparing Quantum Data

In [8]:
"""
One-class Training:
Separation into normal and fraudulent examples
QSVDD will learn what is normal.
"""
normal_indices = np.where(y == 0)[0]
abnormal_indices = np.where(y == 1)[0]
np.random.seed(42)

# Train
# collect 1000 normal examples for training
X_train_normal_indices = np.random.choice(normal_indices, 250, replace=False)
X_train_normal = X_quantum[X_train_normal_indices]
y_train_normal = y[X_train_normal_indices]

# X_train contains only legitimate transactions
X_train = X_train_normal
Y_train = y_train_normal

# Test (balanced)
# The code removes 100 normal examples that were not used in training
remaining_normal_indices = list(set(normal_indices) - set(X_train_normal_indices))
X_test_normal_indices = np.random.choice(remaining_normal_indices, 100, replace=False)
X_test_normal = X_quantum[X_test_normal_indices]
y_test_normal = y[X_test_normal_indices]

# The code removes 100 fraud examples.
X_test_abnormal_indices = np.random.choice(abnormal_indices, 100, replace=False)
X_test_abnormal = X_quantum[X_test_abnormal_indices]
y_test_abnormal = y[X_test_abnormal_indices]

# creates a test set with 200 examples (50% normal, 50% fraud)
X_test = np.concatenate((X_test_normal, X_test_abnormal), axis=0)
Y_test = np.concatenate((y_test_normal, y_test_abnormal), axis=0)

center = np.zeros(latent_dim)
center_train = np.tile(center, (len(X_train), 1))

print(f'X_train shape: {X_train.shape}')
print(f'Y_train shape: {Y_train.shape}')
print(f'X_test_normal shape: {X_test_normal.shape}')
print(f'X_test_abnormal shape: {X_test_abnormal.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'Y_test shape: {Y_test.shape}')
print(f'center_train shape: {center_train.shape}')

X_train shape: (250, 16)
Y_train shape: (250,)
X_test_normal shape: (100, 16)
X_test_abnormal shape: (100, 16)
X_test shape: (200, 16)
Y_test shape: (200,)
center_train shape: (250, 3)


In [9]:
train_Xdata = X_train
train_Ydata = center_train

### QCNN (Quantum Convolutional Neural Network) Ansatz

In [12]:
qcnn_batch_size = 4
qcnn_steps = 2000
qcnn_learning_rate = 0.01

In [13]:
(qcnn_loss_history_matrix,
 qcnn_est_params_matrix,
 qcnn_param_history_matrix,
 qcnn_time_record) = train_five_times(X_train=train_Xdata,
                                        Y_train=train_Ydata,
                                        batch_size=qcnn_batch_size,
                                        learning_rate=qcnn_learning_rate,
                                        steps=qcnn_steps,
                                        ansatz='qcnn'
                                        )
loss_history_f_name = f"../results/training/QCNN/BC_QCNN_B{qcnn_batch_size:02d}S{qcnn_steps}LR{qcnn_learning_rate:.0e}_LOSS_HISTORY_MEAN.npy"
est_params_f_name = f"../results/training/QCNN/BC_QCNN_B{qcnn_batch_size:02d}S{qcnn_steps}LR{qcnn_learning_rate:.0e}_EST_PARAMS_MEAN.npy"
time_f_name = f"../results/training/QCNN/BC_QCNN_B{qcnn_batch_size:02d}S{qcnn_steps}LR{qcnn_learning_rate:.0e}_TIME_MEAN.npy"
np.savetxt(loss_history_f_name, qcnn_loss_history_matrix)
np.savetxt(est_params_f_name, qcnn_est_params_matrix)
np.savetxt(time_f_name, qcnn_time_record)
print("--- All training batches completed ---")

--- Starting training round 1 with seed 859101168 ---


/home/jvfg/Documents/ORG/Repos/QSVDD2/.venv/lib/python3.12/site-packages/autograd/numpy/numpy_vjps.py:943: ComplexWarning: Casting complex values to real discards the imaginary part
  onp.add.at(A, idx, x)


--- Starting training round 2 with seed 713813255 ---
--- Starting training round 3 with seed 148904575 ---
--- Starting training round 4 with seed 2048457818 ---
--- Starting training round 5 with seed 3611616696 ---
--- All training batches completed ---


#### QCNN Training Evaluation

In [14]:
f_name = f"../results/training/QCNN/BC_QCNN_B{qcnn_batch_size:02d}S{qcnn_steps}LR{qcnn_learning_rate:.0e}_EST_PARAMS_MEAN.npy"
params_list = np.loadtxt(f_name)
mean, std = mean_auc(params_list, n_train, X_test, Y_test, center_train, noisy=False, ansatz="qcnn")

print(50*"--")
print(f'for: B{qcnn_batch_size}S{qcnn_steps} | AUC_mean: {mean} | std: {std}')
print(50*"--")

Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.87s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.87s
Test completed in 1.74s | AUC: 0.5134
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.86s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.87s
Test completed in 1.73s | AUC: 0.5248
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.87s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.87s
Test completed in 1.74s | AUC: 0.5763
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.89s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.90s
Test completed in 1.79s | AUC: 0.5356
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.87s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.88s
Test completed in 1.76s | AUC: 0.5527
----------------------------------------------------------------------------------------------------
for: B4S2

### QAE (Quantum AutoEncoder) Ansatz

In [15]:
qae_batch_size = 16
qae_steps = 500
qae_learning_rate = 0.001

#### Five-Run Training & Result Persistence

Run `train_five_times` to train the **QAE** ansatz across 5 independent runs with the configured batch size and step count.

In [16]:
(qae_loss_history_matrix,
 qae_est_params_matrix,
 qae_param_history_matrix,
 qae_time_record) = train_five_times(X_train=train_Xdata,
                                        Y_train=train_Ydata,
                                        batch_size=qae_batch_size,
                                        learning_rate=qae_learning_rate,
                                        steps=qae_steps,
                                        ansatz='qae'
                                        )
loss_history_f_name = f"../results/training/QAE/BC_QAE_B{qae_batch_size:02d}S{qae_steps}LR{qae_learning_rate:.0e}_LOSS_HISTORY_MEAN.npy"
est_params_f_name = f"../results/training/QAE/BC_QAE_B{qae_batch_size:02d}S{qae_steps}LR{qae_learning_rate:.0e}_EST_PARAMS_MEAN.npy"
time_f_name = f"../results/training/QAE/BC_QAE_B{qae_batch_size:02d}S{qae_steps}LR{qae_learning_rate:.0e}_TIME_MEAN.npy"
#np.savetxt(loss_history_f_name, qae_loss_history_matrix)
#np.savetxt(est_params_f_name, qae_est_params_matrix)
#np.savetxt(time_f_name, qae_time_record)
print("--- All training batches completed ---")

--- Starting training round 1 with seed 3847723702 ---
--- Starting training round 2 with seed 1853989119 ---
--- Starting training round 3 with seed 1629211941 ---
--- Starting training round 4 with seed 3034675874 ---
--- Starting training round 5 with seed 3501790717 ---
--- All training batches completed ---


#### QAE Training Evaluation

In [43]:
f_name = f"../results/training/QAE/BC_QAE_B{qae_batch_size:02d}S{qae_steps}LR{qae_learning_rate:.0e}_EST_PARAMS_MEAN.npy"
params_list = np.loadtxt(f_name)
mean, std = mean_auc(params_list, n_train, X_test, Y_test, center_train, noisy=False, ansatz="qae")

print(50*"--")
print(f'for: B{qae_batch_size}S{qae_steps} | AUC_mean: {mean} | std: {std}')
print(50*"--")

Processing class 0 (label 0) | Samples: 100
Finished class 0 in 1.18s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 1.14s
Test completed in 2.33s | AUC: 0.6583
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 1.14s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 1.14s
Test completed in 2.28s | AUC: 0.7904
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 1.14s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 1.15s
Test completed in 2.29s | AUC: 0.8496
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 1.16s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 1.15s
Test completed in 2.31s | AUC: 0.3881
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 1.16s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 1.16s
Test completed in 2.33s | AUC: 0.7812
----------------------------------------------------------------------------------------------------
for: B16S

### LCQHNN (Lean classical-quantum hybrid neural network) Ansatz

The latent space for this ansatz has dimension 5 (vs. 3 for QCNN/QAE), requiring a re-initialized center vector.

In [28]:
center = np.zeros(4)
center_train = np.tile(center, (len(X_train), 1))
lcqhnn_batch_size = 8
lcqhnn_steps = 1000
lcqhnn_learning_rate = 0.01
print(f'center_train shape: {center_train.shape}')
train_Xdata = X_train
train_Ydata = center_train

center_train shape: (250, 4)


#### Five-Run Training & Result Persistence

Run `train_five_times` to train the **QAE** ansatz across 5 independent runs with the configured batch size and step count.

In [29]:
(lcqhnn_loss_history_matrix,
 lcqhnn_est_params_matrix,
 lcqhnn_param_history_matrix,
 lcqhnn_time_record) = train_five_times(X_train=train_Xdata,
                                        Y_train=train_Ydata,
                                        batch_size=lcqhnn_batch_size,
                                        learning_rate=lcqhnn_learning_rate,
                                        steps=lcqhnn_steps,
                                        ansatz='lcqhnn'
                                        )
loss_history_f_name = f"../results/training/LCQHNN/BC_LCQHNN_B{lcqhnn_batch_size:02d}S{lcqhnn_steps}LR{lcqhnn_learning_rate:.0e}_LOSS_HISTORY_MEAN.npy"
est_params_f_name = f"../results/training/LCQHNN/BC_LCQHNN_B{lcqhnn_batch_size:02d}S{lcqhnn_steps}LR{lcqhnn_learning_rate:.0e}_EST_PARAMS_MEAN.npy"
time_f_name = f"../results/training/LCQHNN/BC_LCQHNN_B{lcqhnn_batch_size:02d}S{lcqhnn_steps}LR{lcqhnn_learning_rate:.0e}_TIME_MEAN.npy"
np.savetxt(loss_history_f_name, lcqhnn_loss_history_matrix)
np.savetxt(est_params_f_name, lcqhnn_est_params_matrix)
np.savetxt(time_f_name, lcqhnn_time_record)
print("--- All training batches completed ---")

--- Starting training round 1 with seed 3374572322 ---
--- Starting training round 2 with seed 3741344007 ---
--- Starting training round 3 with seed 190823573 ---
--- Starting training round 4 with seed 4236675335 ---
--- Starting training round 5 with seed 896201761 ---
--- All training batches completed ---


#### LCQHNN Training Evaluation

In [48]:
f_name = f"../results/training/LCQHNN/BC_LCQHNN_B{lcqhnn_batch_size:02d}S{lcqhnn_steps}LR{lcqhnn_learning_rate:.0e}_EST_PARAMS_MEAN.npy"
params_list = np.loadtxt(f_name)
mean, std = mean_auc(params_list, n_train, X_test, Y_test, center_train, noisy=False, ansatz="lcqhnn")

print(50*"--")
print(f'for: B{lcqhnn_batch_size}S{lcqhnn_steps} | AUC_mean: {mean} | std: {std}')
print(50*"--")

Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.21s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.21s
Test completed in 0.42s | AUC: 0.9373
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.20s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.20s
Test completed in 0.41s | AUC: 0.9061
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.20s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.20s
Test completed in 0.40s | AUC: 0.9443
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.20s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.20s
Test completed in 0.40s | AUC: 0.8385
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.20s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.20s
Test completed in 0.40s | AUC: 0.8782
----------------------------------------------------------------------------------------------------
for: B8S1

## Noisy (NISQ-Era) Training

In the noisy setting, the quantum circuits are simulated with a **hardware noise model** that mimics real NISQ (Noisy Intermediate-Scale Quantum) device behavior, including gate errors and decoherence. This evaluates model robustness under realistic quantum hardware conditions.

The same three ansatzes are retrained from scratch under this noisy simulation.